# RedActa — локальное демо

Запускается в **локальном Jupyter** (не Colab). Требования:
- Python-зависимости из `requirements.txt` установлены в активном окружении
- Ollama запущен локально (`ollama serve`) и модель уже загружена

Поддерживаемые операции:
- `replace_point` — изложить пункт в новой редакции
- `repeal_point` — признать пункт утратившим силу
- `insert_point` — вставить новый пункт
- `append_section_item` — дополнить подпункт или элемент
- `replace_phrase_globally` — заменить или удалить фразу по документу
- `replace_appendix_block` — изложить в новой редакции структурно однозначный блок приложения

In [1]:
# ── Ячейка 1: Пути к файлам — отредактируйте перед запуском ───────────────
from pathlib import Path

# Корень репозитория (папка, где лежат src/, config/, redactions/)
REPO_DIR = Path("C:/Users/appan/Downloads/RedActa_1/RedActa-main").resolve().parent.parent

# Пути к документам (абсолютные или относительно текущей папки)
BASE_DOCX      = REPO_DIR / "C:/Users/appan/Downloads/RedActa_1/RedActa-main/redactions/04/Приказ Минфина России от 11.09.2020 N 191н.docx"
AMENDMENT_DOCX = REPO_DIR / "C:/Users/appan/Downloads/RedActa_1/RedActa-main/redactions/04/изм_Приказ Минфина России от 03.10.2025 N 141н.docx"

# Конфиг модели
MODELS_CONFIG = REPO_DIR / "C:/Users/appan/Downloads/RedActa_1/RedActa-main/config/models.example.json"

# ID кейса — определяет папку вывода artifacts/<CASE_ID>/
CASE_ID = "04"

# Проверка
for label, p in [("base", BASE_DOCX), ("amendment", AMENDMENT_DOCX), ("config", MODELS_CONFIG)]:
    status = "OK" if p.exists() else "НЕ НАЙДЕН"
    print(f"{label:12s}: {p}  [{status}]")

base        : C:\Users\appan\Downloads\RedActa_1\RedActa-main\redactions\04\Приказ Минфина России от 11.09.2020 N 191н.docx  [OK]
amendment   : C:\Users\appan\Downloads\RedActa_1\RedActa-main\redactions\04\изм_Приказ Минфина России от 03.10.2025 N 141н.docx  [OK]
config      : C:\Users\appan\Downloads\RedActa_1\RedActa-main\config\models.example.json  [OK]


In [6]:
# ── Ячейка 2: Python path ──────────────────────────────────────────────────
import sys

src_dir = str(REPO_DIR / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print(f"src: {src_dir}")

src: C:\Users\appan\Downloads\src


In [7]:
# ── Ячейка 3: Проверка Ollama ──────────────────────────────────────────────
import urllib.request
import json as _json

try:
    with urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=3) as resp:
        tags = _json.loads(resp.read())
    models = [m["name"] for m in tags.get("models", [])]
    print(f"Ollama готов. Загружено моделей: {len(models)}")
    for m in models:
        print(f"  - {m}")
except Exception as exc:
    print(f"Ollama недоступен: {exc}")
    print("Запустите в терминале: ollama serve")

Ollama готов. Загружено моделей: 8
  - qwen2.5-coder:14b
  - qwen2.5-coder:1.5b-base
  - qwen35:4bit
  - qwen35:5bit
  - qwen35:8bit
  - hf.co/ai-sage/GigaChat3.1-10B-A1.8B-GGUF:Q4_K_M
  - deepseek-v4-flash:cloud
  - qwen3.6:27b


In [8]:
# ── Ячейка 4: Запуск пайплайна ────────────────────────────────────────────
import json
from graph_pipeline.colab_runner import run_uploaded_pair

result = run_uploaded_pair(
    base_docx=BASE_DOCX,
    amendment_docx=AMENDMENT_DOCX,
    workspace_root=REPO_DIR,
    models_config=MODELS_CONFIG,
    case_id=CASE_ID,
)

validation = result.get("validation", {})
print("\nСводка валидации:")
print(json.dumps(validation, ensure_ascii=False, indent=2))

output_doc = result.get("output_doc")
print(f"\nВыходной документ: {output_doc}")

status = "valid" if validation.get("is_valid") else "invalid"
print(f"Статус: {status}")

total_markers = sum(len(step.get("revision_markers", [])) for step in result.get("steps", []))
print(f"Маркеры редакции: {total_markers} вставлено")

[01:03:16][SDK Pipeline][04] ============================================================
[01:03:16][SDK Pipeline][04] Запуск SDK pipeline
[01:03:16][SDK Pipeline][04]   topology: standard_single
[01:03:16][SDK Pipeline][04]   amendments: 1
[01:03:16][SDK Pipeline][04]   bases: 1
[01:03:16][SDK Pipeline][04]   artifacts: C:\Users\appan\Downloads\artifacts\04
[01:03:16][SDK Pipeline][04] ============================================================
[01:03:16][SDK Pipeline][04] --- Шаг 1/4: Аналитики ---
[01:03:16][AmendmentAnalyzer] start 1/1: изм_Приказ Минфина России от 03.10.2025 N 141н.docx
[01:03:16][BaseAnalyzer] start: Приказ Минфина России от 11.09.2020 N 191н.docx
[01:03:16][BaseAnalyzer] end: Приказ Минфина России от 11.09.2020 N 191н.docx (header_blocks=2, elapsed=0.2с)
[01:03:34][AmendmentAnalyzer] end   1/1: изм_Приказ Минфина России от 03.10.2025 N 141н.docx (intents=7, elapsed=17.9с)
[01:03:34][SDK Pipeline][04]   amendment docs: 1
[01:03:34][SDK Pipeline][04]   intents: 7

In [9]:
# ── Ячейка 5: Открыть выходной документ (Windows / macOS / Linux) ──────────
import subprocess
import platform

if output_doc:
    p = Path(output_doc)
    if p.exists():
        system = platform.system()
        if system == "Windows":
            subprocess.Popen(["start", "", str(p)], shell=True)
        elif system == "Darwin":
            subprocess.Popen(["open", str(p)])
        else:
            subprocess.Popen(["xdg-open", str(p)])
        print(f"Открываю: {p}")
    else:
        print(f"Файл не найден: {p}")
else:
    print("Сначала запустите ячейку 4.")

Открываю: C:\Users\appan\Downloads\artifacts\04\working_step_1.docx


## Диагностика

- *Файл НЕ НАЙДЕН в ячейке 1* — укажите правильные пути к `.docx`.
- *Ollama недоступен (ячейка 3)* — запустите `ollama serve` в отдельном терминале.
- *`ModuleNotFoundError: graph_pipeline`* — убедитесь, что ячейка 2 выполнена, и что `REPO_DIR` указывает на корень репозитория.
- *Статус `invalid`* — смотрите сводку валидации и папку `artifacts/<CASE_ID>/` с debug JSON.
- *Маркеры редакции: 0* — пайплайн не нашёл однозначных точек привязки; это норма при частичном разрешении intents.